# Code Ownership, notebook 6: the final test (once) and the production artifact

The chosen model is refit on all development functions (settings chosen by inner grouped folds over the development set) and scored **once** on the 6 final-test functions.
Then it is refit on all 36 functions for the artifact the agent loads. The test result comes only from the development-fit model.

In [ ]:
import sys, warnings, json, pickle
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../../helpers").resolve()))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
import ko_data as K, ko_models as M
from core import metrics, group_bootstrap, paired_diff_bootstrap, qwk, band
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 90)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
d = K.load()
dev = d[d.role == "dev"].reset_index(drop=True)
test = d[d.role == "test"].reset_index(drop=True)
y, groups, folds = dev.score.to_numpy(), dev.snippet_id.to_numpy(), dev.fold.to_numpy()
def ci(p, stat):
    return group_bootstrap(y, p, groups, stat, n=1000)
def sp(a, b):
    return spearmanr(a, b).correlation
def mae(a, b):
    return float(np.mean(np.abs(a - b)))
def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

In [ ]:
import joblib
choice = json.loads((RESULTS / "model_choice.json").read_text(encoding="utf-8"))
prod = choice["production"]
print(prod)
Hd, Ht = K.hand_features(dev), K.hand_features(test)
if prod["kind"] == "encoder":
    Xd = K.feature_sets(K.embeddings(prod["encoder"], dev), Hd)[prod["feature_set"]]
    Xt = K.feature_sets(K.embeddings(prod["encoder"], test), Ht)[prod["feature_set"]]
else:
    Xd, Xt = Hd.to_numpy(float), Ht.to_numpy(float)
Xd, Xt = np.asarray(Xd, float), np.asarray(Xt, float)
head = prod["head"]
NH = Hd.shape[1] if prod["feature_set"].endswith("+H") else 0
from concurrent.futures import ProcessPoolExecutor
with ProcessPoolExecutor(10) as pool:
    params, _ = M.tune_one(head, Xd, y, groups, 0, pool, NH)
print("settings chosen on the development set:", params)
pt = M.fit_predict(head, params, Xd, y, Xt, 0)
yt = test.score.to_numpy()

In [ ]:
tg = test.snippet_id.to_numpy()
def tci(stat):
    return group_bootstrap(yt, pt, tg, stat, n=2000)
m = metrics(yt, pt)
res = {"n answers": len(yt), "n functions": len(set(tg)), "MAE": m["MAE"], "RMSE": m["RMSE"], "Spearman": m["spearman"], "Pearson": m["pearson"], "QWK (1-5 bands)": m["band_qwk"], "ICC(2,1)": m["ICC2_1"]}
print({k: round(v, 3) for k, v in res.items()})
print("95% intervals (grouped bootstrap over the 6 functions): MAE", np.round(tci(mae), 3), "Spearman", np.round(tci(lambda a, b: sp(a, b)), 3))

In [ ]:
# the same baselines, fitted on the development functions only, scored on the same test answers
from sklearn.linear_model import LinearRegression
rows = [{"model": "production model", "MAE": mae(pt, yt), "Spearman": sp(pt, yt)}, {"model": "training mean", "MAE": mae(np.full(len(yt), y.mean()), yt), "Spearman": float("nan")}]
for col, name in (("n_words", "answer length"), ("identifier_recall", "identifier recall"), ("focus_overlap", "overlap with the asked-about fact")):
    lr = LinearRegression().fit(Hd[[col]], y); p = np.clip(lr.predict(Ht[[col]]), 0, 1)
    rows.append({"model": name, "MAE": mae(p, yt), "Spearman": sp(p, yt)})
pd.DataFrame(rows).round(3)

In [ ]:
t2 = test.assign(pred=pt, err=np.abs(pt - yt))
print(t2.groupby("question_type").agg(n=("err", "size"), MAE=("err", "mean")).round(3))
fig, ax = plt.subplots(figsize=(5, 4)); sns.stripplot(data=t2, x="rating", y=1 + 4 * t2.pred, ax=ax, jitter=0.15); ax.plot([0, 4], [1, 5], "k--", lw=1); ax.set_ylabel("predicted rating (1-5)"); ax.set_xlabel("written level"); plt.show()

Sensitivity to the label: the same predictions against the independent rating and against the mean of the two ratings.

In [ ]:
out = {}
for col in ("score", "score_independent", "score_mean"):
    yy = test[col].to_numpy(); out[col] = {"MAE": mae(pt, yy), "Spearman": sp(pt, yy)}
pd.DataFrame(out).T.round(3)

## Production artifact

Refit on all 36 functions (settings chosen on all of them), plus the 10th and 90th percentile of the development out-of-fold residuals as the reported band.

In [ ]:
allX = np.vstack([Xd, Xt]); ally = np.concatenate([y, yt]); allg = np.concatenate([groups, tg])
with ProcessPoolExecutor(10) as pool:
    params_all, _ = M.tune_one(head, allX, ally, allg, 0, pool, NH)
final = M.make(head, params_all, 0).fit(allX, ally)
P = pickle.load(open(RESULTS / "tuning_preds.pkl", "rb"))
B = pickle.load(open(RESULTS / "baselines_oof.pkl", "rb"))
oof = P[choice["encoder_model"]]["pred"] if prod["kind"] == "encoder" else B[prod["key"]]
resid = y - oof
MODEL = Path("../../../model").resolve(); MODEL.mkdir(exist_ok=True)
joblib.dump(final, MODEL / "ko_head.joblib")
card = {"encoder": prod["encoder"], "feature_set": prod["feature_set"], "head": head, "params": params_all, "features_version": "ko_features/2",
        "trained_on": "36 standard-library functions, 216 assistant-written answers (no student data, no human ratings)",
        "dev_oof": {"MAE": mae(oof, y), "Spearman": sp(oof, y)}, "final_test": {k: float(v) for k, v in res.items()},
        "residual_quantiles": [float(np.percentile(resid, 10)), float(np.percentile(resid, 90))]}
(MODEL / "ko_model.json").write_text(json.dumps(card, indent=1), encoding="utf-8")
card

## Reading the results

_Reading of the results: written after the run._